# Research_2DGS: Anisotropic BRDF in 2DGS for Inverse Rendering
This notebook sets up the environment and runs training for the Anisotropic GGX PBR model on 2D Gaussian Splatting.

### Step 1: Check GPU and CUDA Toolkit

In [ ]:
!nvidia-smi
!nvcc --version

### Step 2: Clone Code recursively

In [ ]:
!git clone --recursive https://github.com/shInNei/Research_2DGS.git /content/Research_2DGS
%cd /content/Research_2DGS
!git fetch origin
!git checkout forward-sg-palette-sobel

### Step 3: Install dependencies and compile CUDA submodules

In [ ]:
%cd /content/Research_2DGS
!pip install plyfile opencv-python lpips trimesh open3d tqdm mediapy

# Patch simple-knn compiling issue (FLT_MAX not found on modern compilers)
!python -c "with open('submodules/simple-knn/simple_knn.cu', 'r+') as f: c = f.read(); f.seek(0); f.write('#include <cfloat>\n' + c)"

# Compile custom CUDA rasterizer and KNN submodules
!pip install submodules/diff-surfel-rasterization
!pip install submodules/simple-knn

### Step 4: Download Dataset

In [ ]:
%cd /content/Research_2DGS
!mkdir -p data
%cd data

# Download TensoIR Lego Synthetic Dataset from Zenodo
!wget -O lego.zip "https://zenodo.org/records/7880113/files/lego.zip?download=1"
!unzip -q lego.zip -d lego
!rm lego.zip

%cd /content/Research_2DGS

### Step 5: Start Training!

In [ ]:
%cd /content/Research_2DGS
!python train.py -s data/lego --model_path output/tensoir_lego --eval

### Step 6: Render Trajectory and Evaluate Metrics
Run evaluation to render views, export material maps (Albedo, Normals, Roughness, Metallic), generate video trajectories, and compute comparative metrics.

In [ ]:
# Render testing images and video trajectories
!python render.py -m output/tensoir_lego --render_path --skip_mesh

# Evaluate PSNR, SSIM, LPIPS metrics
!python metrics.py -m output/tensoir_lego

### Step 7: View Metrics and Material Videos
Display computed metrics and visualize material trajectory videos (Color, Albedo, Normal, Roughness, Metallic).

In [ ]:
import json
import os
import mediapy as media

# 1. Print metrics table
metrics_file = "output/tensoir_lego/results.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        data = json.load(f)
    print("=" * 40)
    print("       Evaluation Metrics (ours_30000)   ")
    print("=" * 40)
    for method, metrics in data.items():
        print(f"Method: {method}")
        for k, v in metrics.items():
            print(f"  {k:<10}: {v:.5f}")
    print("=" * 40)
else:
    print("Metrics file not found. Make sure the evaluation script ran successfully.")

# 2. Display videos
traj_dir = "output/tensoir_lego/traj/ours_30000"
for tag in ["color", "albedo", "normal", "roughness", "metallic", "depth"]:
    video_path = os.path.join(traj_dir, f"render_traj_{tag}.mp4")
    if os.path.exists(video_path):
        print(f"\nDisplaying {tag.upper()} trajectory video:")
        media.show_video(video_path, height=400)
    else:
        print(f"Video {video_path} not found.")

### Step 8: Backup Outputs to Google Drive (Prevent Data Loss)

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Copy the training output (Gaussians checkpoints, logs, and renders) to Drive
!mkdir -p /content/drive/MyDrive/Research_2DGS_outputs/
!cp -r /content/Research_2DGS/output/* /content/drive/MyDrive/Research_2DGS_outputs/